# RSNA Knee — Phase 1–4 完整训练管线

> 在 Kaggle 上一键运行全部 4 个 Phase 的训练 + 集成推理。
>
> **当前默认**: Phase 1 (EfficientNetV2-S) + Ensemble 推理 (~2h)
>
> ## 需要挂载的 Kaggle Inputs

| Input | 类型 | 说明 |
|-------|------|------|
| Competition Dataset | 自动 | RSNA 2026 Knee (train_series, test_series, train.csv) |
| Source Code Bundle | Dataset | 本项目源码 (models/, datasets/, losses/, utils.py) |
| Pseudo Labels | Dataset | `pseudo_labels.csv` (NLP 生成的 4349 条伪标签) |

## Phase 路线图

| Phase | 模型 | 输入 | 参数量 | VRAM | 默认 |
|-------|------|------|--------|------|------|
| **P1** | EfficientNetV2-S 2.5D | 5-slice Sagittal | ~21M | ~4GB | ✅ 运行 |
| **P2** | TriPlane (Shared EffNetV2-S) | Sag+Cor+Ax 5-slice | ~23M | ~6GB | ⏭ 跳过 |
| **P3** | ResNet3D-18 | 32-slice 3D volume | ~33M | ~8GB | ⏭ 跳过 |
| **P4** | ConvNeXt-S / Swin-T | 5-slice Sagittal | ~50M each | ~6GB | ⏭ 跳过 |
| **Ens** | 加权平均集成 | 可用模型预测合并 | — | ~4GB | ✅ 运行 |

## 运行控制

修改 Cell 1 中的 `RUN_PHASES` 字典选择要运行的 Phase。

In [ ]:
# ============================================================
# Cell 1: 环境 & 配置
# ============================================================
from __future__ import annotations
import gc, json, os, random, sys, time, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── 随机种子 ───────────────────────────────────────────────
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ── 设备 ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 4  # Kaggle 上可用 2, 如报错改为 0
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)")

# ── 12 类标签 ──────────────────────────────────────────────
TARGETS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# ── ⚙️ 运行控制: 设为 False 跳过对应 Phase ──────────────────
RUN_PHASES = {
    "phase1_efficientnet": True,    # EfficientNetV2-S 2.5D  ← 当前只跑这个
    "phase2_triplane":     False,   # Tri-Plane 三平面融合
    "phase3_resnet3d":     False,   # 3D ResNet-18 volume
    "phase4_convnext":     False,   # ConvNeXt-S
    "phase4_swin":         False,   # Swin-T
    "ensemble_submission": True,    # 最终集成推理
}

# ── 共用超参数 ─────────────────────────────────────────────
CFG = dict(
    image_size=384, num_slices=5, volume_depth=32, volume_size=128,
    lr=2e-4, weight_decay=1e-4, dropout=0.3,
    focal_gamma=2.0, focal_alpha=0.25,
    patience=5, min_delta=0.001,
)
print(f"Image: {CFG['image_size']}×{CFG['image_size']}  Slices: {CFG['num_slices']}")
print(f"Volume: {CFG['volume_depth']}×{CFG['volume_size']}×{CFG['volume_size']}")

# 打印运行计划
active = [k for k, v in RUN_PHASES.items() if v]
print(f"\n📋 运行计划: {', '.join(active)}")
print(f"   跳过的 Phase: {', '.join(k for k, v in RUN_PHASES.items() if not v)}")

# ── Kaggle 输出目录与兼容性辅助函数 ──────────────────────────
OUTPUT_DIR = Path("/kaggle/working")
CKPT_DIR = OUTPUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def make_grad_scaler():
    """兼容不同 Kaggle PyTorch 版本的 AMP scaler。"""
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def amp_context():
    """CPU 环境下安全禁用 CUDA autocast。"""
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    from contextlib import nullcontext
    return nullcontext()

def build_model(model_cls, **kwargs):
    """优先加载预训练权重；Kaggle 离线且无缓存时回退到随机初始化。"""
    try:
        return model_cls(**kwargs)
    except Exception as exc:
        if not kwargs.get("pretrained", False):
            raise
        print(f"⚠ 预训练权重不可用 ({type(exc).__name__}: {exc})，改用随机初始化")
        fallback = dict(kwargs)
        fallback["pretrained"] = False
        return model_cls(**fallback)


In [ ]:
# ============================================================
# Cell 2: 文件发现 & 源码导入（固定路径，避免递归扫描卡住）
# ============================================================
INPUT = Path("/kaggle/input")
COMPETITION_CANDIDATES = [
    INPUT / "competitions" / "rsna-knee-abnormality-detection",
    INPUT / "rsna-knee-abnormality-detection",
]
COMP_ROOT = next((p for p in COMPETITION_CANDIDATES if p.is_dir()), None)
if COMP_ROOT is None:
    raise FileNotFoundError(
        "找不到官方竞赛目录。请挂载 rsna-knee-abnormality-detection Competition Dataset。"
    )

def require_file(name: str) -> Path:
    path = COMP_ROOT / name
    if not path.is_file():
        raise FileNotFoundError(f"缺少官方文件: {path}")
    return path

TRAIN_CSV = require_file("train.csv")
SERIES_CSV = require_file("train_series.csv")
TEST_SERIES_CSV = require_file("test_series.csv")
SAMPLE_CSV = require_file("sample_submission.csv")
DICOM_ROOT = COMP_ROOT / "train_series"
TEST_DICOM_ROOT = COMP_ROOT / "test_series"
if not DICOM_ROOT.is_dir():
    raise FileNotFoundError(f"找不到训练 DICOM 目录: {DICOM_ROOT}")

# 用户提供的 Kaggle Dataset 路径；同时兼容 Kaggle 常见的简化挂载路径。
PSEUDO_DATASET_CANDIDATES = [
    INPUT / "datasets" / "charlottemorandi" / "pseudo-labels-csv",
    INPUT / "pseudo-labels-csv",
]
SOURCE_DATASET_CANDIDATES = [
    INPUT / "datasets" / "charlottemorandi" / "kaggle-source-bundle-zip",
    INPUT / "kaggle-source-bundle-zip",
]

# 只检查挂载 Dataset 的浅层常用位置，不再 INPUT.rglob 全盘扫描。
def shallow_find(filename: str, required: bool = True) -> Path | None:
    candidates = []
    for dataset_root in INPUT.iterdir():
        if dataset_root == COMP_ROOT or dataset_root.name == "competitions":
            continue
        candidates.extend([
            dataset_root / filename,
            dataset_root / "data" / filename,
            dataset_root / "data" / "metadata" / filename,
        ])
    hit = next((p for p in candidates if p.is_file()), None)
    if required and hit is None:
        raise FileNotFoundError(f"找不到 {filename}，请把它作为 Kaggle Dataset 挂载")
    return hit

PSEUDO_CSV = next(
    (root / "pseudo_labels.csv" for root in PSEUDO_DATASET_CANDIDATES
     if (root / "pseudo_labels.csv").is_file()),
    None,
)
if PSEUDO_CSV is None:
    PSEUDO_CSV = shallow_find("pseudo_labels.csv")

# 用项目特征文件定位源码根目录；若 Dataset 内只有 ZIP，则安全解压。
import zipfile

def is_source_root(path: Path) -> bool:
    return (
        (path / "models" / "efficientnet25d.py").is_file()
        and (path / "datasets" / "dataset.py").is_file()
        and (path / "utils.py").is_file()
    )

def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination_resolved != target and destination_resolved not in target.parents:
                raise ValueError(f"源码 ZIP 包含不安全路径: {member.filename}")
        archive.extractall(destination)

source_root = None
source_roots = [p for p in SOURCE_DATASET_CANDIDATES if p.is_dir()]
for dataset_root in source_roots:
    for candidate in (dataset_root, dataset_root / "CVproject_RSNA-knee-detection"):
        if is_source_root(candidate):
            source_root = candidate
            break
    if source_root is not None:
        break

if source_root is None:
    zip_path = next(
        (p for root in source_roots for p in (
            root / "kaggle_source_bundle.zip",
            root / "source_bundle.zip",
            root / "CVproject_RSNA-knee-detection.zip",
        ) if p.is_file()),
        None,
    )
    if zip_path is None:
        zip_path = next((p for root in source_roots for p in root.glob("*.zip")), None)
    if zip_path is not None:
        extracted_root = OUTPUT_DIR / "source_bundle"
        marker = extracted_root / ".extracted_ok"
        if not marker.exists():
            safe_extract_zip(zip_path, extracted_root)
            marker.touch()
        candidates = [extracted_root, extracted_root / "CVproject_RSNA-knee-detection"]
        candidates.extend(p.parent.parent for p in extracted_root.glob("*/models/efficientnet25d.py"))
        source_root = next((p for p in candidates if is_source_root(p)), None)

if source_root is None:
    raise FileNotFoundError(
        "源码 Dataset 中未找到可用项目或 ZIP；需要 models/、datasets/、losses/ 和 utils.py。"
    )
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

print("=== 文件发现 ===")
print(f"  Competition:    {COMP_ROOT}")
print(f"  Train CSV:      {TRAIN_CSV}")
print(f"  Series CSV:     {SERIES_CSV}")
print(f"  DICOM root:     {DICOM_ROOT}")
print(f"  Test DICOM:     {TEST_DICOM_ROOT}")
print(f"  Pseudo labels:  {PSEUDO_CSV}")
print(f"  Source code:    {source_root}")

from datasets.dataset import Knee25DDataset
from datasets.triplane_dataset import TriPlaneDataset
from datasets.volume_dataset import VolumeDataset
from datasets.pseudo_labels import PseudoLabelLoader
from datasets.dicom_loader import read_dicom_series
from models.efficientnet25d import EfficientNetV2S25D
from models.triplane import TriPlaneModel
from models.resnet3d import ResNet3DModel
from models.convnext import ConvNeXt25D
from models.swin import Swin25D
from losses.focal_bce import FocalBCELoss
from utils import (
    TARGET_COLUMNS, aggregate_to_study,
    compute_macro_auc, compute_per_class_auc, format_per_class_auc,
)

if list(TARGET_COLUMNS) != TARGETS:
    raise ValueError("Notebook TARGETS 与源码 TARGET_COLUMNS 顺序不一致")
print("\n✓ 所有模块导入成功")

class FastKneeSeries25DDataset(Dataset):
    """每个 series 只产生一个样本，避免为每张中心切片重复解码整个 DICOM 序列。

    训练时每个 epoch 随机中心切片，验证/推理使用中心切片。与原 Dataset
    的返回字段完全兼容，但通常把每 epoch 的 DICOM 解码次数降低 15–40 倍。
    """
    def __init__(self, series_df, labels_df, dicom_root, planes=None,
                 image_size=384, slice_count=5, is_train=True, **_):
        self.root = Path(dicom_root)
        self.image_size = int(image_size)
        self.slice_count = int(slice_count)
        self.half = self.slice_count // 2
        self.is_train = bool(is_train)
        planes = planes or ["Sagittal", "Coronal", "Axial"]

        labels = labels_df.copy()
        if "StudyInstanceUID" not in labels.columns:
            labels = labels.reset_index()
        labels["StudyInstanceUID"] = labels["StudyInstanceUID"].astype(str)
        self.label_map = labels.set_index("StudyInstanceUID")[TARGETS].astype(np.float32)

        frame = series_df.copy()
        frame["StudyInstanceUID"] = frame["StudyInstanceUID"].astype(str)
        frame["SeriesInstanceUID"] = frame["SeriesInstanceUID"].astype(str)
        frame = frame[frame["Anatomical_Plane"].isin(planes)]
        frame = frame.drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])

        self.samples = []
        for row in frame.itertuples(index=False):
            uid = str(row.StudyInstanceUID)
            if uid not in self.label_map.index:
                continue
            series_uid = str(row.SeriesInstanceUID)
            series_dir = self.root / uid / series_uid
            if series_dir.is_dir():
                self.samples.append((uid, series_uid, str(row.Anatomical_Plane), str(series_dir)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        uid, series_uid, plane, series_dir = self.samples[index]
        labels = self.label_map.loc[uid].to_numpy(np.float32, copy=True)
        try:
            volume = read_dicom_series(series_dir, plane=plane, image_size=self.image_size)
            n = len(volume)
            if n == 0:
                raise ValueError("empty DICOM series")
            if self.is_train and n > 1:
                center = int(np.random.randint(0, n))
            else:
                center = n // 2
            ids = [min(max(center + offset, 0), n - 1)
                   for offset in range(-self.half, self.half + 1)]
            image = torch.from_numpy(volume[ids].copy())
        except Exception as exc:
            # 记录有限数量的坏 series；返回零图保证长训练不中断。
            if not hasattr(self, "_error_count"):
                self._error_count = 0
            if self._error_count < 5:
                print(f"⚠ DICOM 读取失败 {series_uid}: {type(exc).__name__}: {exc}")
            self._error_count += 1
            image = torch.zeros(self.slice_count, self.image_size, self.image_size)
        return {
            "image": image,
            "labels": torch.from_numpy(labels),
            "study_uid": uid,
            "plane": plane,
        }


def loader_options():
    options = {
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
    }
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return options


In [ ]:
# ============================================================
# Cell 3: 伪标签加载 & Gold-Label 验证集
# ============================================================
# 所有 Phase 共用: pseudo=train, gold=val (固定 split)

pseudo_loader = PseudoLabelLoader(pseudo_csv=str(PSEUDO_CSV))
stats = pseudo_loader.stats()

print("=== 伪标签统计 ===")
print(f"  Total:          {stats['total']:,}")
by_conf = stats["by_confidence_overall"]
for level in ["HIGH", "HIGH_PLUS_MEDIUM", "MEDIUM"]:
    if level in by_conf:
        pct = by_conf[level] / stats["total"] * 100
        print(f"  {level:<20s} {by_conf[level]:,} ({pct:.1f}%)")

print(f"\n  类别分布 (全部 4349 studies):")
for k, v in sorted(stats["class_balance"].items(), key=lambda x: -x[1]):
    bar = "█" * int(v / max(stats["class_balance"].values()) * 30)
    print(f"    {k:<20s} {v:5d}  {bar}")

# ── 合并 pseudo (train) + gold (val) ────────────────────────
train_labels, val_labels = pseudo_loader.merge_with_gold(
    gold_csv=str(TRAIN_CSV), confidence="HIGH", val_from_gold=True,
)
if train_labels.empty:
    print("⚠ HIGH 过滤后没有训练样本，自动放宽为 HIGH_PLUS_MEDIUM")
    train_labels, val_labels = pseudo_loader.merge_with_gold(
        gold_csv=str(TRAIN_CSV), confidence="HIGH_PLUS_MEDIUM", val_from_gold=True,
    )
if train_labels.empty:
    raise ValueError("伪标签过滤后训练集为空，请检查 pred_* 与 conf_* 字段")
if val_labels.empty:
    raise ValueError("Gold 验证集为空，请检查官方 train.csv 的标签字段")

# PseudoLabelLoader 返回 StudyInstanceUID 索引，而 Dataset 接口要求它是普通列。
def labels_for_dataset(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "StudyInstanceUID" not in out.columns:
        out = out.reset_index()
    required = ["StudyInstanceUID", *TARGETS]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"标签表缺少字段: {missing}")
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    return out[required]

train_labels = labels_for_dataset(train_labels)
val_labels = labels_for_dataset(val_labels)
print(f"\n  Train (pseudo HIGH): {len(train_labels):,} studies")
print(f"  Valid (gold):         {len(val_labels):,} studies")

# ── 加载 series 元数据 ──────────────────────────────────────
series_df = pd.read_csv(SERIES_CSV, dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
print(f"\n  Series 元数据: {len(series_df):,} rows")
print(f"  DICOM 目录存在: {DICOM_ROOT.exists()}")

---
## Phase 1 — EfficientNetV2-S 2.5D Baseline

- **Backbone**: EfficientNetV2-S (`timm`)
- **输入**: 5 张相邻 Sagittal 切片堆叠为 5 通道 → 384×384
- **训练**: freeze backbone 5 epochs → unfreeze 全量微调
- **损失**: Focal BCE (γ=2, α=0.25)
- **目标**: macro AUC ≈ 0.649

In [ ]:
# ============================================================
# Cell 5: Phase 1 — EfficientNetV2-S 训练
# ============================================================
if RUN_PHASES["phase1_efficientnet"]:
    print("=" * 55)
    print("Phase 1: EfficientNetV2-S 2.5D Baseline")
    print("=" * 55)

    # ── Dataset ────────────────────────────────────────────
    ds_kwargs = dict(
        dicom_root=str(DICOM_ROOT), planes=["Sagittal"],
        image_size=CFG["image_size"], slice_count=CFG["num_slices"],
        fluid_sensitive_only=False, fat_suppression_only=False,
    )
    train_ds_p1 = FastKneeSeries25DDataset(series_df, train_labels, is_train=True, **ds_kwargs)
    val_ds_p1 = FastKneeSeries25DDataset(series_df, val_labels, is_train=False, **ds_kwargs)
    print(f"  Train samples: {len(train_ds_p1):,}  |  Val samples: {len(val_ds_p1):,}")

    if len(train_ds_p1) == 0 or len(val_ds_p1) == 0:
        raise ValueError("train_ds_p1/val_ds_p1 为空，请检查平面名称、UID 类型和 DICOM 路径")
    loader_kw = loader_options()
    train_loader_p1 = DataLoader(train_ds_p1, batch_size=8, shuffle=True, **loader_kw)
    val_loader_p1 = DataLoader(val_ds_p1, batch_size=8, shuffle=False, **loader_kw)

    # ── Model ──────────────────────────────────────────────
    model_p1 = build_model(EfficientNetV2S25D,
        in_channels=5, num_classes=12, pretrained=True,
        dropout=CFG["dropout"],
    ).to(DEVICE)
    if DEVICE.type == "cuda":
        model_p1 = model_p1.to(memory_format=torch.channels_last)
    # 前 5 epochs 只训练分类头，显著降低初期反向传播开销。
    for parameter in model_p1.backbone.parameters():
        parameter.requires_grad = False

    n_p = sum(p.numel() for p in model_p1.parameters()) / 1e6
    n_t = sum(p.numel() for p in model_p1.parameters() if p.requires_grad) / 1e6
    print(f"  Params: {n_p:.1f}M total, {n_t:.1f}M trainable")

    # ── Training setup ─────────────────────────────────────
    criterion = FocalBCELoss(gamma=CFG["focal_gamma"], alpha=CFG["focal_alpha"])
    optimizer = torch.optim.AdamW(
        model_p1.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6,
    )
    scaler = make_grad_scaler()

    # ── Training loop ──────────────────────────────────────
    best_auc, patience_ctr = 0.0, 0
    ckpt_dir = CKPT_DIR
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    history_p1 = []
    t_start = time.time()

    EPOCHS = 20
    for epoch in range(EPOCHS):
        t0 = time.time()
        if epoch == 5:
            for parameter in model_p1.backbone.parameters():
                parameter.requires_grad = True
            print("  ✓ Epoch 6: backbone 已解冻，开始全量微调")
        model_p1.train()
        total_loss = 0.0
        for batch in tqdm(train_loader_p1, desc=f"P1 E{epoch+1}/{EPOCHS}", leave=False):
            x = batch["image"].to(DEVICE, non_blocking=True)
            if DEVICE.type == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)
            y = batch["labels"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                loss = criterion(model_p1(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_p1.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(x)

        scheduler.step()

        # ── Validation ──────────────────────────────────
        model_p1.eval()
        val_logits_dict = defaultdict(list)
        val_targets_dict = {}
        with torch.inference_mode():
            for batch in val_loader_p1:
                x = batch["image"].to(DEVICE, non_blocking=True)
                if DEVICE.type == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)
                logits = model_p1(x).float().cpu().numpy()
                for uid, z, lbl in zip(batch["study_uid"], logits, batch["labels"].numpy()):
                    val_logits_dict[uid].append(z)
                    val_targets_dict[uid] = lbl

        uids = sorted(val_logits_dict)
        z_val = np.stack([np.mean(val_logits_dict[u], 0) for u in uids])
        y_val = np.stack([val_targets_dict[u] for u in uids])
        val_auc = compute_macro_auc(y_val, z_val)
        per_class = compute_per_class_auc(y_val, z_val)

        elapsed = time.time() - t0
        avg_sec = (time.time() - t_start) / (epoch + 1)
        remaining = avg_sec * (EPOCHS - epoch - 1)
        eta_str = f"{remaining/60:.0f}min" if remaining < 3600 else f"{remaining/3600:.1f}h"

        lr_now = optimizer.param_groups[0]["lr"]
        info = {"epoch": epoch + 1, "loss": total_loss / max(len(train_ds_p1), 1),
                "val_auc": val_auc, "lr": lr_now, "time": f"{elapsed:.0f}s", "eta": eta_str}
        print(f"  P1 E{epoch+1:3d}: loss={info['loss']:.4f}  val_auc={val_auc:.4f}  "
              f"lr={lr_now:.2e}  ⏱ {elapsed:.0f}s  ETA {eta_str}")
        print(f"         per-class: {format_per_class_auc(per_class)}")
        history_p1.append(info)

        if val_auc > best_auc + CFG["min_delta"]:
            best_auc = val_auc
            patience_ctr = 0
            torch.save(
                {"model": model_p1.state_dict(), "epoch": epoch, "auc": best_auc},
                ckpt_dir / "efficientnetv2_s_best.pt",
            )
        else:
            patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    total_h = (time.time() - t_start) / 3600
    print(f"\n✓ Phase 1 完成: best_auc={best_auc:.4f}  耗时={total_h:.1f}h")

    # 保存历史
    with open(Path("/kaggle/working") / "history_phase1.json", "w") as f:
        json.dump({"best_auc": best_auc, "history": history_p1}, f, indent=2)

    # 释放 VRAM
    del model_p1, train_ds_p1, val_ds_p1, train_loader_p1, val_loader_p1
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("⏭  跳过 Phase 1 (RUN_PHASES['phase1_efficientnet'] = False)")

---
## Phase 2 — Tri-Plane 三平面融合

- **Backbone**: 共享 EfficientNetV2-S (三平面共用参数)
- **输入**: Sagittal + Coronal + Axial 各 5 切片 → 三平面 → concat fusion → head
- **缺失处理**: 无 Cor/Ax 平面的 study 用 learnable `missing_emb` 替换
- **梯度累积**: bs=4, accum=4 (effective bs=16)
- **目标**: macro AUC ≈ 0.661 (预期高于单平面)

In [ ]:
# ============================================================
# Cell 7: Phase 2 — Tri-Plane 训练
# ============================================================
if RUN_PHASES["phase2_triplane"]:
    print("=" * 55)
    print("Phase 2: Tri-Plane 三平面融合训练")
    print("=" * 55)

    # ── Dataset ────────────────────────────────────────────
    train_ds_p2 = TriPlaneDataset(
        series_df=series_df, labels_df=train_labels,
        dicom_root=str(DICOM_ROOT),
        image_size=CFG["image_size"], slice_count=CFG["num_slices"],
        planes=["Sagittal", "Coronal", "Axial"], is_train=True,
    )
    val_ds_p2 = TriPlaneDataset(
        series_df=series_df, labels_df=val_labels,
        dicom_root=str(DICOM_ROOT),
        image_size=CFG["image_size"], slice_count=CFG["num_slices"],
        planes=["Sagittal", "Coronal", "Axial"], is_train=False,
    )
    print(f"  Train: {len(train_ds_p2):,} slices  |  Val: {len(val_ds_p2):,} slices")

    if len(train_ds_p2) == 0 or len(val_ds_p2) == 0:
        raise ValueError("train_ds_p2/val_ds_p2 为空，请检查平面名称、UID 类型和 DICOM 路径")
    BATCH_SIZE = 4
    ACCUM_STEPS = 4
    loader_kw = dict(pin_memory=DEVICE.type == "cuda")
    train_loader_p2 = DataLoader(train_ds_p2, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, **loader_kw)
    val_loader_p2 = DataLoader(val_ds_p2, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, **loader_kw)

    # ── Model ──────────────────────────────────────────────
    model_p2 = build_model(TriPlaneModel,
        in_channels=5, num_classes=12, pretrained=True,
        dropout=CFG["dropout"], shared_backbone=True, fusion="concat",
    ).to(DEVICE)

    n_p = sum(p.numel() for p in model_p2.parameters()) / 1e6
    n_t = sum(p.numel() for p in model_p2.parameters() if p.requires_grad) / 1e6
    print(f"  Params: {n_p:.1f}M total, {n_t:.1f}M trainable")

    # ── Training setup ─────────────────────────────────────
    criterion = FocalBCELoss(gamma=CFG["focal_gamma"], alpha=CFG["focal_alpha"])
    optimizer = torch.optim.AdamW(
        model_p2.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6,
    )
    scaler = make_grad_scaler()

    # ── Training loop ──────────────────────────────────────
    best_auc, patience_ctr = 0.0, 0
    ckpt_dir = CKPT_DIR
    history_p2 = []
    t_start = time.time()
    n_batches = len(train_loader_p2)
    EPOCHS = 50

    for epoch in range(EPOCHS):
        t0 = time.time()
        model_p2.train()
        total_loss = 0.0
        optimizer.zero_grad()

        for b_idx, batch in enumerate(train_loader_p2):
            sag = batch["sag"].to(DEVICE)
            cor = batch["cor"].to(DEVICE)
            ax = batch["ax"].to(DEVICE)
            y = batch["labels"].to(DEVICE)

            with amp_context():
                loss = criterion(model_p2(sag, cor, ax), y) / ACCUM_STEPS

            scaler.scale(loss).backward()

            is_step = (b_idx + 1) % ACCUM_STEPS == 0
            is_last = (b_idx + 1) == n_batches
            if is_step or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model_p2.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += loss.item() * ACCUM_STEPS

        scheduler.step()

        # ── Validation ──────────────────────────────────
        model_p2.eval()
        val_logits_dict = defaultdict(list)
        val_targets_dict = {}
        with torch.inference_mode():
            for batch in val_loader_p2:
                sag = batch["sag"].to(DEVICE)
                cor = batch["cor"].to(DEVICE)
                ax = batch["ax"].to(DEVICE)
                logits = model_p2(sag, cor, ax).float().cpu().numpy()
                for uid, z, lbl in zip(batch["study_uid"], logits, batch["labels"].numpy()):
                    val_logits_dict[uid].append(z)
                    val_targets_dict[uid] = lbl

        uids = sorted(val_logits_dict)
        z_val = np.stack([np.mean(val_logits_dict[u], 0) for u in uids])
        y_val = np.stack([val_targets_dict[u] for u in uids])
        val_auc = compute_macro_auc(y_val, z_val)
        per_class = compute_per_class_auc(y_val, z_val)

        elapsed = time.time() - t0
        avg_sec = (time.time() - t_start) / (epoch + 1)
        remaining = avg_sec * (EPOCHS - epoch - 1)
        eta_str = f"{remaining/60:.0f}min" if remaining < 3600 else f"{remaining/3600:.1f}h"

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  P2 E{epoch+1:3d}: loss={total_loss/max(n_batches,1):.4f}  val_auc={val_auc:.4f}  "
              f"lr={lr_now:.2e}  ⏱{elapsed:.0f}s  ETA{eta_str}  eff_bs={BATCH_SIZE}×{ACCUM_STEPS}")
        print(f"         per-class: {format_per_class_auc(per_class)}")
        history_p2.append({"epoch": epoch + 1, "loss": total_loss / max(n_batches, 1),
                           "val_auc": val_auc})

        if val_auc > best_auc + CFG["min_delta"]:
            best_auc = val_auc
            patience_ctr = 0
            torch.save(
                {"model": model_p2.state_dict(), "epoch": epoch, "auc": best_auc},
                ckpt_dir / "triplane_best.pt",
            )
        else:
            patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    total_h = (time.time() - t_start) / 3600
    print(f"\n✓ Phase 2 完成: best_auc={best_auc:.4f}  耗时={total_h:.1f}h")
    with open(Path("/kaggle/working") / "history_phase2.json", "w") as f:
        json.dump({"best_auc": best_auc, "history": history_p2}, f, indent=2)

    del model_p2, train_ds_p2, val_ds_p2, train_loader_p2, val_loader_p2
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("⏭  跳过 Phase 2 (RUN_PHASES['phase2_triplane'] = False)")

---
## Phase 3 — ResNet3D-18 3D Volumetric

- **Backbone**: ResNet3D-18 (torchvision/timm 3D)
- **输入**: 32 张连续 Sagittal 切片 → [1, 32, 128, 128]
- **关键优化**:
  - Gradient Checkpointing (用计算换 VRAM)
  - batch_size=1, gradient_accumulation=8 (effective bs=8)
  - AMP fp16
  - 体积 resize 到 128×128 (减少计算量)
- **目标**: 验证 3D 管线在 8GB VRAM 上可跑通，macro AUC ≈ 0.642

In [ ]:
# ============================================================
# Cell 9: Phase 3 — 3D ResNet 训练
# ============================================================
if RUN_PHASES["phase3_resnet3d"]:
    print("=" * 55)
    print("Phase 3: ResNet3D-18 3D Volumetric Training")
    print("=" * 55)

    # ── Dataset ────────────────────────────────────────────
    train_ds_p3 = VolumeDataset(
        series_df=series_df, labels_df=train_labels,
        dicom_root=str(DICOM_ROOT),
        volume_depth=CFG["volume_depth"], volume_size=CFG["volume_size"],
        plane="Sagittal", is_train=True,
    )
    val_ds_p3 = VolumeDataset(
        series_df=series_df, labels_df=val_labels,
        dicom_root=str(DICOM_ROOT),
        volume_depth=CFG["volume_depth"], volume_size=CFG["volume_size"],
        plane="Sagittal", is_train=False,
    )
    print(f"  Train volumes: {len(train_ds_p3):,}  |  Val volumes: {len(val_ds_p3):,}")

    if len(train_ds_p3) == 0 or len(val_ds_p3) == 0:
        raise ValueError("train_ds_p3/val_ds_p3 为空，请检查平面名称、UID 类型和 DICOM 路径")
    BATCH_SIZE = 1
    ACCUM_STEPS = 8
    loader_kw = dict(pin_memory=DEVICE.type == "cuda")
    train_loader_p3 = DataLoader(train_ds_p3, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=1, **loader_kw)
    val_loader_p3 = DataLoader(val_ds_p3, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=1, **loader_kw)

    # ── Model ──────────────────────────────────────────────
    model_p3 = build_model(ResNet3DModel,
        in_channels=1, num_classes=12, pretrained=True,
        dropout=CFG["dropout"], use_grad_checkpoint=True,
    ).to(DEVICE)

    n_p = sum(p.numel() for p in model_p3.parameters()) / 1e6
    n_t = sum(p.numel() for p in model_p3.parameters() if p.requires_grad) / 1e6
    print(f"  Params: {n_p:.1f}M total, {n_t:.1f}M trainable")
    print(f"  Grad ckpt: ON  |  AMP: {USE_AMP}  |  Effective BS: {BATCH_SIZE}×{ACCUM_STEPS}={BATCH_SIZE*ACCUM_STEPS}")

    # ── Training setup ─────────────────────────────────────
    criterion = FocalBCELoss(gamma=CFG["focal_gamma"], alpha=CFG["focal_alpha"])
    optimizer = torch.optim.AdamW(
        model_p3.parameters(), lr=1e-4, weight_decay=CFG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6,
    )
    scaler = make_grad_scaler()

    # ── Training loop ──────────────────────────────────────
    best_auc, patience_ctr = 0.0, 0
    ckpt_dir = CKPT_DIR
    history_p3 = []
    t_start = time.time()
    n_batches = len(train_loader_p3)
    EPOCHS = 40

    for epoch in range(EPOCHS):
        t0 = time.time()
        model_p3.train()
        total_loss = 0.0
        optimizer.zero_grad()

        for b_idx, batch in enumerate(train_loader_p3):
            volume = batch["volume"].to(DEVICE)  # [B, 1, D, H, W]
            y = batch["labels"].to(DEVICE)

            with amp_context():
                loss = criterion(model_p3(volume), y) / ACCUM_STEPS

            scaler.scale(loss).backward()

            is_step = (b_idx + 1) % ACCUM_STEPS == 0
            is_last = (b_idx + 1) == n_batches
            if is_step or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model_p3.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += loss.item() * ACCUM_STEPS

        scheduler.step()

        # ── Validation ──────────────────────────────────
        model_p3.eval()
        all_logits, all_labels = [], []
        with torch.inference_mode():
            for batch in val_loader_p3:
                volume = batch["volume"].to(DEVICE)
                all_logits.append(model_p3(volume).float().cpu().numpy())
                all_labels.append(batch["labels"].numpy())

        z_val = np.concatenate(all_logits)
        y_val = np.concatenate(all_labels)
        val_auc = compute_macro_auc(y_val, z_val)
        per_class = compute_per_class_auc(y_val, z_val)

        elapsed = time.time() - t0
        avg_sec = (time.time() - t_start) / (epoch + 1)
        remaining = avg_sec * (EPOCHS - epoch - 1)
        eta_str = f"{remaining/60:.0f}min" if remaining < 3600 else f"{remaining/3600:.1f}h"
        vram = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  P3 E{epoch+1:3d}: loss={total_loss/max(n_batches,1):.4f}  val_auc={val_auc:.4f}  "
              f"lr={lr_now:.2e}  VRAM={vram:.1f}GB  ⏱{elapsed:.0f}s  ETA{eta_str}")
        print(f"         per-class: {format_per_class_auc(per_class)}")
        history_p3.append({"epoch": epoch + 1, "loss": total_loss / max(n_batches, 1),
                           "val_auc": val_auc, "vram_gb": vram})

        if val_auc > best_auc + CFG["min_delta"]:
            best_auc = val_auc
            patience_ctr = 0
            torch.save(
                {"model": model_p3.state_dict(), "epoch": epoch, "auc": best_auc},
                ckpt_dir / "resnet3d_best.pt",
            )
        else:
            patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    total_h = (time.time() - t_start) / 3600
    print(f"\n✓ Phase 3 完成: best_auc={best_auc:.4f}  耗时={total_h:.1f}h")
    with open(Path("/kaggle/working") / "history_phase3.json", "w") as f:
        json.dump({"best_auc": best_auc, "history": history_p3}, f, indent=2)

    del model_p3, train_ds_p3, val_ds_p3, train_loader_p3, val_loader_p3
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("⏭  跳过 Phase 3 (RUN_PHASES['phase3_resnet3d'] = False)")

---
## Phase 4 — Multi-Backbone Ensemble 训练

- **Backbones**: ConvNeXt-S (768d), Swin-T (768d)
- **输入**: 5-slice Sagittal 2.5D (384×384)
- **训练**: 各自独立训练 (无需联合训练的 VRAM 压力)
- **推理**: 与 P1 EffNet + P2 TriPlane + P3 ResNet3D 加权平均集成

> 💡 EfficientNetV2-S 在 P1 中已训练，此处只训练 ConvNeXt 和 Swin。

In [ ]:
# ============================================================
# Cell 11: Phase 4 — ConvNeXt-S / Swin-T 训练
# ============================================================
def train_single_backbone(name, model_cls, model_kwargs, train_labels, val_labels,
                          series_df, batch_size=8, epochs=50, lr=2e-4):
    """训练单个 backbone, 返回 best_auc."""
    print(f"\n{'─'*50}")
    print(f"训练: {name}")

    ds_kwargs = dict(
        dicom_root=str(DICOM_ROOT), planes=["Sagittal"],
        image_size=CFG["image_size"], slice_count=CFG["num_slices"],
        fluid_sensitive_only=False, fat_suppression_only=False,
    )
    train_ds = FastKneeSeries25DDataset(series_df, train_labels, is_train=True, **ds_kwargs)
    val_ds = FastKneeSeries25DDataset(series_df, val_labels, is_train=False, **ds_kwargs)
    print(f"  Train: {len(train_ds):,} slices  |  Val: {len(val_ds):,} slices")

    loader_kw = loader_options()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              **loader_kw)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            **loader_kw)

    model = build_model(model_cls, **model_kwargs).to(DEVICE)
    n_p = sum(p.numel() for p in model.parameters()) / 1e6
    n_t = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"  Params: {n_p:.1f}M total, {n_t:.1f}M trainable")

    criterion = FocalBCELoss(gamma=CFG["focal_gamma"], alpha=CFG["focal_alpha"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6,
    )
    scaler = make_grad_scaler()

    ckpt_dir = CKPT_DIR
    best_auc, patience_ctr = 0.0, 0
    t_start = time.time()

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["labels"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(x)

        scheduler.step()

        model.eval()
        val_logits_dict = defaultdict(list)
        val_targets_dict = {}
        with torch.inference_mode():
            for batch in val_loader:
                x = batch["image"].to(DEVICE, non_blocking=True)
                logits = model(x).float().cpu().numpy()
                for uid, z, lbl in zip(batch["study_uid"], logits, batch["labels"].numpy()):
                    val_logits_dict[uid].append(z)
                    val_targets_dict[uid] = lbl

        uids = sorted(val_logits_dict)
        z_val = np.stack([np.mean(val_logits_dict[u], 0) for u in uids])
        y_val = np.stack([val_targets_dict[u] for u in uids])
        val_auc = compute_macro_auc(y_val, z_val)
        per_class = compute_per_class_auc(y_val, z_val)

        elapsed = time.time() - t0
        avg_sec = (time.time() - t_start) / (epoch + 1)
        remaining = avg_sec * (epochs - epoch - 1)
        eta_str = f"{remaining/60:.0f}min" if remaining < 3600 else f"{remaining/3600:.1f}h"

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  [{name}] E{epoch+1:3d}: loss={total_loss/max(len(train_ds),1):.4f}  "
              f"val_auc={val_auc:.4f}  lr={lr_now:.2e}  ⏱{elapsed:.0f}s  ETA{eta_str}")
        print(f"         per-class: {format_per_class_auc(per_class)}")

        if val_auc > best_auc + CFG["min_delta"]:
            best_auc = val_auc
            patience_ctr = 0
            ckpt_name = f"{name.lower().replace('-','_')}_best.pt"
            torch.save(
                {"model": model.state_dict(), "epoch": epoch, "auc": best_auc, "arch": name},
                ckpt_dir / ckpt_name,
            )
        else:
            patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                print(f"  [{name}] Early stopping at epoch {epoch+1}")
                break

    total_h = (time.time() - t_start) / 3600
    print(f"✓ {name} 完成: best_auc={best_auc:.4f}  耗时={total_h:.1f}h")
    del model, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_auc


# ── ConvNeXt-S ─────────────────────────────────────────────
p4_results = {}

if RUN_PHASES["phase4_convnext"]:
    print("=" * 55)
    print("Phase 4a: ConvNeXt-S 2.5D")
    print("=" * 55)
    p4_results["convnext_small"] = train_single_backbone(
        "ConvNeXt-S", ConvNeXt25D,
        {"in_channels": 5, "num_classes": 12, "pretrained": True, "dropout": CFG["dropout"]},
        train_labels, val_labels, series_df, batch_size=6, epochs=50,
    )
else:
    print("⏭  跳过 ConvNeXt-S")

# ── Swin-T ─────────────────────────────────────────────────
if RUN_PHASES["phase4_swin"]:
    print("=" * 55)
    print("Phase 4b: Swin-T 2.5D")
    print("=" * 55)
    p4_results["swin_tiny"] = train_single_backbone(
        "Swin-T", Swin25D,
        {"in_channels": 5, "num_classes": 12, "pretrained": True, "dropout": CFG["dropout"]},
        train_labels, val_labels, series_df, batch_size=4, epochs=50,
    )
else:
    print("⏭  跳过 Swin-T")

print(f"\nPhase 4 结果: {p4_results}")

---
## Ensemble 集成推理 & Submission

加载所有已训练 checkpoint → 逐 slice 推理 → top-K mean 聚合 → 加权平均 sigmoid 概率 → submission CSV

### 集成权重 (基于各模型 val AUC 动态调整)

默认权重: EffNet 0.25 | TriPlane 0.30 | ResNet3D 0.20 | ConvNeXt-S 0.15 | Swin-T 0.10

In [ ]:
# ============================================================
# Cell 13: Ensemble — 推理 & 生成 Submission
# ============================================================
if RUN_PHASES["ensemble_submission"]:
    print("=" * 55)
    print("Ensemble Inference & Submission")
    print("=" * 55)

    ckpt_dir = CKPT_DIR

    # ── 发现可用 checkpoint ───────────────────────────────
    model_registry = [
        ("effnet",   EfficientNetV2S25D, "efficientnetv2_s_best.pt", 0.25,
         {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": CFG["dropout"]}),
        ("triplane", TriPlaneModel,       "triplane_best.pt",           0.30,
         {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": CFG["dropout"],
          "shared_backbone": True, "fusion": "concat"}),
        ("resnet3d", ResNet3DModel,      "resnet3d_best.pt",           0.20,
         {"in_channels": 1, "num_classes": 12, "pretrained": False, "dropout": CFG["dropout"],
          "use_grad_checkpoint": False}),
        ("convnext", ConvNeXt25D,         "convnext_small_best.pt",     0.15,
         {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": CFG["dropout"]}),
        ("swin",     Swin25D,             "swin_tiny_best.pt",          0.10,
         {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": CFG["dropout"]}),
    ]

    # 加载可用模型
    ensemble_models = []  # [(name, model, weight), ...]
    for mname, mcls, fname, weight, kwargs in model_registry:
        ckpt_path = ckpt_dir / fname
        if not ckpt_path.exists():
            print(f"  ⚠ {mname}: checkpoint 不存在 ({fname}) — 跳过")
            continue

        model = mcls(**kwargs).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("model", ckpt.get("model_state_dict", ckpt))
        state = {k.replace("module.", ""): v for k, v in state.items()}
        model.load_state_dict(state, strict=False)
        model.eval()

        val_auc = ckpt.get("auc", ckpt.get("val_auc", 0))
        ensemble_models.append((mname, model, weight))
        n_p = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"  ✓ {mname:<12s} weight={weight:.2f}  val_auc={val_auc:.4f}  params={n_p:.1f}M")

    if not ensemble_models:
        print("\n⚠ 无可用 checkpoint！生成全 0.5 的 dummy submission。")
        # 直接生成 dummy submission，无需加载测试集
        if SAMPLE_CSV.exists():
            sample_sub = pd.read_csv(SAMPLE_CSV)
            submission = sample_sub.copy()
            submission[TARGETS] = 0.5
            submission.to_csv("/kaggle/working/submission.csv", index=False)
            print(f"✓ Dummy submission 已保存: /kaggle/working/submission.csv")
            print(f"  Shape: {submission.shape}")
        else:
            print("⚠ sample_submission.csv 不存在，无法生成 submission")
    else:
        # 归一化权重
        total_w = sum(w for _, _, w in ensemble_models)
        ensemble_models = [(n, m, w / total_w) for n, m, w in ensemble_models]
        print(f"\n 归一化后权重: {', '.join(f'{n}={w:.3f}' for n,_,w in ensemble_models)}")

        # ── 构建 Test Dataset ──────────────────────────────
        if TEST_DICOM_ROOT is not None and TEST_DICOM_ROOT.exists() and TEST_SERIES_CSV.exists():
            test_series_df = pd.read_csv(TEST_SERIES_CSV, dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
            test_study_uids = sorted(test_series_df["StudyInstanceUID"].unique())
            print(f"\n测试集: {len(test_study_uids):,} studies, {len(test_series_df):,} series")

            # 构建 2.5D dataset (EffNet/ConvNeXt/Swin 共用)
            dummy_labels = pd.DataFrame({
                "StudyInstanceUID": test_study_uids,
                **{col: 0 for col in TARGETS},
            })

            test_ds_2d = FastKneeSeries25DDataset(
                series_df=test_series_df, labels_df=dummy_labels,
                dicom_root=str(TEST_DICOM_ROOT), planes=["Sagittal"],
                image_size=CFG["image_size"], slice_count=CFG["num_slices"],
                is_train=False,
            )
            print(f"  2.5D test samples: {len(test_ds_2d):,} slices")

            loader_kw = loader_options()
            test_loader_2d = DataLoader(test_ds_2d, batch_size=8, shuffle=False, **loader_kw)

            # ── 逐模型推理 ────────────────────────────────
            # study_uid → {model_name → [logits_from_slices]}
            study_slice_logits: dict[str, dict[str, list[np.ndarray]]] = \
                defaultdict(lambda: defaultdict(list))

            model_names_2d = {"effnet", "convnext", "swin"}
            print(f"\n推理中...")

            with torch.inference_mode():
                for batch in tqdm(test_loader_2d, desc="Ensemble inference", unit="batch"):
                    x = batch["image"].to(DEVICE, non_blocking=True)
                    uids = batch["study_uid"]

                    with amp_context():
                        for mname, model, _ in ensemble_models:
                            if mname not in model_names_2d:
                                continue
                            logits = model(x).float().cpu().numpy()
                            for j, uid in enumerate(uids):
                                study_slice_logits[str(uid)][mname].append(logits[j])

            # ── TriPlane 推理 (如果有) ────────────────────
            if any(n == "triplane" for n, _, _ in ensemble_models):
                print("  TriPlane inference...")
                test_ds_tri = TriPlaneDataset(
                    series_df=test_series_df, labels_df=dummy_labels,
                    dicom_root=str(TEST_DICOM_ROOT),
                    image_size=CFG["image_size"], slice_count=CFG["num_slices"],
                    planes=["Sagittal", "Coronal", "Axial"], is_train=False,
                )
                test_loader_tri = DataLoader(test_ds_tri, batch_size=4, shuffle=False,
                                             num_workers=NUM_WORKERS, **loader_kw)
                tri_model = next((m for n, m, _ in ensemble_models if n == "triplane"), None)
                if tri_model is not None:
                    for batch in tqdm(test_loader_tri, desc="TriPlane infer", leave=False):
                        sag = batch["sag"].to(DEVICE)
                        cor = batch["cor"].to(DEVICE)
                        ax = batch["ax"].to(DEVICE)
                        with amp_context():
                            logits = tri_model(sag, cor, ax).float().cpu().numpy()
                        for j, uid in enumerate(batch["study_uid"]):
                            study_slice_logits[str(uid)]["triplane"].append(logits[j])

            # ── 3D 推理 (如果有) ──────────────────────────
            if any(n == "resnet3d" for n, _, _ in ensemble_models):
                print("  3D inference...")
                test_ds_3d = VolumeDataset(
                    series_df=test_series_df, labels_df=dummy_labels,
                    dicom_root=str(TEST_DICOM_ROOT),
                    volume_depth=CFG["volume_depth"], volume_size=CFG["volume_size"],
                    plane="Sagittal", is_train=False,
                )
                test_loader_3d = DataLoader(test_ds_3d, batch_size=1, shuffle=False,
                                            num_workers=1, **loader_kw)
                r3d_model = next((m for n, m, _ in ensemble_models if n == "resnet3d"), None)
                if r3d_model is not None:
                    for batch in tqdm(test_loader_3d, desc="3D infer", leave=False):
                        vol = batch["volume"].to(DEVICE)
                        with amp_context():
                            logits = r3d_model(vol).float().cpu().numpy()
                        for j, uid in enumerate(batch["study_uid"]):
                            study_slice_logits[str(uid)]["resnet3d"].append(logits[j])

            # ── Slice → Study 聚合 & 加权平均 ────────────
            print(f"\n聚合 & 加权平均...")
            model_weight_map = {n: w for n, _, w in ensemble_models}
            ensemble_probs = {}

            for uid in test_study_uids:
                uid_str = str(uid)
                model_logits = study_slice_logits.get(uid_str, {})

                if not model_logits:
                    ensemble_probs[uid_str] = np.full(12, 0.5, dtype=np.float32)
                    continue

                weighted_sum = np.zeros(12, dtype=np.float64)
                weight_total = 0.0

                for mname, logit_list in model_logits.items():
                    stacked = np.stack(logit_list)  # [K, 12]
                    k = max(1, int(len(stacked) * 0.25))
                    top_vals = np.sort(stacked, axis=0)[-k:]
                    study_logit = top_vals.mean(axis=0)  # [12]

                    w = model_weight_map.get(mname, 0.0)
                    weighted_sum += study_logit * w
                    weight_total += w

                if weight_total > 0:
                    weighted_sum /= weight_total

                probs = 1.0 / (1.0 + np.exp(-np.clip(weighted_sum, -30, 30)))
                ensemble_probs[uid_str] = probs.astype(np.float32)

            # ── 生成 Submission ──────────────────────────
            sample_sub = pd.read_csv(SAMPLE_CSV) if SAMPLE_CSV.exists() else \
                pd.DataFrame({"StudyInstanceUID": test_study_uids})

            output = np.zeros((len(sample_sub), 12), dtype=np.float32)
            for i, uid in enumerate(sample_sub["StudyInstanceUID"]):
                output[i] = ensemble_probs.get(str(uid), np.full(12, 0.5))

            submission = sample_sub.copy()
            submission[TARGETS] = np.clip(output, 0.001, 0.999)

            # 质量控制
            assert not submission.isna().any().any(), "NaN in submission!"
            submission.to_csv("/kaggle/working/submission.csv", index=False)

            print(f"\n✓ Submission 已保存: /kaggle/working/submission.csv")
            print(f"  Shape: {submission.shape}")
            print(f"  Range: [{output.min():.4f}, {output.max():.4f}]")
            print(f"  Mean:  {output.mean():.4f}")
            print(f"\n前 5 行预览:")
            display(submission.head())
        else:
            print("\n⚠ 无测试集 DICOM (test_series/ 目录不存在)")
            print("  推理需要挂载完整的 Competition Dataset (含 test_series)")
else:
    print("⏭  跳过 Ensemble 推理")

In [ ]:
# ============================================================
# Cell 14: 训练总结 & 清理
# ============================================================
print("=" * 60)
print("RSNA Knee — Phase 1-4 训练总结")
print("=" * 60)

# 汇总所有历史
all_best = {}
for phase_name, history_file in [
    ("Phase1_EffNet", "history_phase1.json"),
    ("Phase2_TriPlane", "history_phase2.json"),
    ("Phase3_ResNet3D", "history_phase3.json"),
]:
    hp = Path("/kaggle/working") / history_file
    if hp.exists():
        with open(hp) as f:
            data = json.load(f)
        all_best[phase_name] = data["best_auc"]
        print(f"  {phase_name:<25s} best AUC = {data['best_auc']:.4f}")

# 检查 checkpoint
ckpt_dir = Path("/kaggle/working/checkpoints")
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob("*.pt"))
    print(f"\n  Checkpoints ({len(ckpts)}):")
    for c in ckpts:
        size_mb = c.stat().st_size / 1024**2
        print(f"    {c.name} ({size_mb:.1f} MB)")

# VRAM
if DEVICE.type == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"\n  Peak VRAM: {peak:.1f} GB")

print(f"\n{'='*60}")
print("训练完成! 输出文件:")
print("  /kaggle/working/submission.csv        — 最终提交")
print("  /kaggle/working/checkpoints/           — 模型权重")
print("  /kaggle/working/history_phase*.json    — 训练历史")
print("=" * 60)

---
## 使用说明

### Kaggle 运行步骤

1. **创建源码 Dataset**: 将项目 `models/`, `datasets/`, `losses/`, `utils.py` 打包上传
2. **上传伪标签 Dataset**: 包含 `pseudo_labels.csv` (NLP 生成的 4349 条)
3. **创建 Kaggle Notebook**: 上传此 notebook
4. **挂载 Inputs**: Competition Dataset (自动) + 源码 bundle + 伪标签
5. **选择运行的 Phase**: 修改 Cell 1 中的 `RUN_PHASES` 字典
6. **Run All**

### 分阶段运行建议

```python
# 只跑 Phase 1 (最快，~2h)
RUN_PHASES = {k: False for k in RUN_PHASES}
RUN_PHASES["phase1_efficientnet"] = True
RUN_PHASES["ensemble_submission"] = True

# 完整 5 模型训练 (~12h on P100)
RUN_PHASES = {k: True for k in RUN_PHASES}
```

### 常见问题

| 现象 | 原因 | 解决 |
|------|------|------|
| `找不到 utils.py` | 源码 bundle 未正确挂载 | 检查 Kaggle Dataset 包含 `utils.py` 且路径正确 |
| `找不到 pseudo_labels.csv` | 伪标签 Dataset 路径嵌套太深 | Cell 2 已使用 `rglob` 递归搜索, 确保挂载了正确的 Dataset |
| TriPlaneDataset 为 0 | 三平面 DICOM 不齐全 | 正常 — Cor/Ax 缺失的 study 会被跳过 |
| 3D VRAM OOM | T4 16GB 不够 | 降低 `volume_size` 到 96 或 `volume_depth` 到 24 |
| Ensemble 推理无结果 | 没有 checkpoint | 至少运行一个 Phase 训练 |
| `AttributeError: 'NoneType' object has no attribute 'exists'` | test_series DICOM 目录未挂载 | 确保 Competition Dataset 已挂载 (含 test_series/) |
| DataLoader 卡死 / 报错 | num_workers 默认值不兼容 | 修改 Cell 1 中 `NUM_WORKERS = 0` 禁用多进程加载 |